### Libraries

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from plotnine import ggplot, aes, geom_point, scale_x_log10, scale_y_continuous, scale_color_manual, labs, theme_bw, theme, element_text
import matplotlib.gridspec as gridspec
import os
import pickle

group = "Panthera_uncia"
#group = "Ceratotherium_simum"
#group = "Rhinoceros_unicornis"
#group = "Boselaphus_tragocamelus"

ref = "Tragelaphus_eurycerus_isaaci"


if os.getcwd().startswith("/home/lakrids"):
    path_prefix = "/home/lakrids/GenomeDK"
else:
    path_prefix = "/faststorage/project/"
with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/population_list.txt") as f:
        pops = [line.strip() for line in f]
population_name = pops[0]
with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/parameters_{group}.pkl", "rb") as file:
    parameters = pickle.load(file)

if group == "Boselaphus_tragocamelus":
    df_regions = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/data/{ref}/ref/regions_{ref}_updated.txt", delimiter="\t")
else: 
    df_regions = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/data/{group}/ref/regions_{group}_updated.txt", delimiter="\t")

    
with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/{population_name}_filtered.txt") as f:
    samples = [line.strip() for line in f if line.strip()]


### File Input & First Data Impression

In [3]:
df_stats = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/data/{group}/analysis_input/smcpp/masked_regions/mask_stats_{group}.txt")

chromosomes = df_regions[
        (df_regions["female_ploidy"] == 2) &
        (df_regions["male_ploidy"] == 2) &
        (df_regions["end"] > 20000000)
    ]["region"].tolist()

# Build a contig length dict from the regions df
contig_lengths = df_regions.set_index("region")["end"].to_dict()

df_stats = df_stats[["sample"] + chromosomes]
# Divide each chromosome column by its contig lengths
for chrom in df_stats.columns:
    if chrom in contig_lengths:
        df_stats[chrom] = df_stats[chrom] / contig_lengths[chrom]
df_stats

,sample,NW_026057576.1,NW_026057577.1,NC_064811.1,NC_064808.1,NC_064814.1,NC_064813.1,NC_064816.1,NW_026057578.1,NW_026057579.1,...,NW_026057581.1,NC_064815.1,NW_026057582.1,NC_064809.1,NC_064807.1,NW_026057584.1,NW_026057585.1,NW_026057586.1,NW_026057588.1,NW_026057589.1
0,SAMN02086968,0.073170,0.079134,0.071982,0.118429,0.162474,0.123382,0.113263,0.111784,0.104777,...,0.100318,0.161154,0.094226,0.098909,0.125234,0.086490,0.129805,0.108866,0.157234,0.154468
1,SAMN13911153,0.051843,0.053544,0.052660,0.062503,0.067274,0.060556,0.056959,0.059468,0.058210,...,0.052251,0.068789,0.056911,0.057069,0.063565,0.054390,0.058346,0.057350,0.083300,0.061751
2,SAMN15801464,0.016128,0.018879,0.017682,0.024310,0.020662,0.020696,0.018636,0.020458,0.017107,...,0.021068,0.023110,0.019503,0.018852,0.024210,0.017828,0.018127,0.016851,0.034018,0.020430
3,SAMN38638828,0.208280,0.181431,0.209816,0.199890,0.126385,0.195041,0.198684,0.188786,0.155027,...,0.162136,0.126026,0.161859,0.165784,0.195816,0.189430,0.196773,0.182869,0.133057,0.154324
4,SAMN38638830,0.113580,0.112481,0.112979,0.117714,0.118296,0.117902,0.111986,0.167140,0.107869,...,0.114258,0.110710,0.117322,0.116095,0.119424,0.113924,0.116485,0.111713,0.123096,0.110505
5,SAMN38638834,0.055259,0.055011,0.056699,0.060391,0.053092,0.056152,0.052454,0.076508,0.047959,...,0.054284,0.052048,0.055181,0.054763,0.059966,0.054977,0.052004,0.052854,0.065540,0.053967
6,SAMN38638837,0.278166,0.265264,0.298509,0.284545,0.157985,0.248422,0.280853,0.267833,0.200471,...,0.226044,0.172434,0.288289,0.292314,0.259247,0.269391,0.269128,0.242990,0.173384,0.219184
7,SAMN38768039,0.097299,0.100219,0.101538,0.115460,0.126626,0.111083,0.108776,0.111670,0.107010,...,0.100230,0.122507,0.107213,0.106235,0.114875,0.103174,0.110482,0.107520,0.133112,0.114709
8,SAMN40633821,0.079187,0.079160,0.080023,0.095495,0.137605,0.098292,0.090677,0.102535,0.097098,...,0.085699,0.136466,0.079763,0.085026,0.102612,0.085603,0.095586,0.084451,0.126402,0.116990
9,SAMN40633822,0.061855,0.059498,0.062219,0.077181,0.084024,0.067109,0.074541,0.072355,0.061290,...,0.066728,0.084025,0.067775,0.071347,0.072991,0.063929,0.063775,0.055813,0.098226,0.086258


In [ ]:
df_stats.to_csv(f"{path_prefix}/megaFauna/sa_megafauna/data/{group}/analysis_input/smcpp/masked_regions/mask_stats_{group}.txt")

In [4]:
genus_list      = ["Loxodonta", "Elephas", "Boselaphus", "Panthera", "Rhinoceros", "Ceratotherium", "Diceros"]
#genus_list = ["Elephas"]

data            = pd.concat([pd.read_table(f) for f in [f"{path_prefix}/megaFauna/sa_megafauna/metadata/samples_{genus}.txt" for genus in genus_list]], ignore_index=True) # add all SRR accession from genus_list in a single data frame
data            = data.reset_index(drop=True)
ref_folders     = sorted(set(data.REFERENCE_FOLDER)) # list of references needed to map the SRR accessions

references      = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/metadata/references.txt")
references      = references.loc[references.REFERENCE_FOLDER.isin(ref_folders)] # filter references to keep only the ones needed to map the SRR accessions
references      = references.reset_index(drop=True)

## make data frame that contains names of species-specific folders and the reference folders used to map the species
species_and_refs = pd.DataFrame({"FOLDER": data.FOLDER, "REFERENCE_FOLDER": data.REFERENCE_FOLDER, "GVCF_FOLDER": [data.GENUS.iloc[jj] + "_" + data.SPECIES.iloc[jj] for jj in range(data.shape[0])]}).drop_duplicates()
species_and_refs = species_and_refs.reset_index(drop=True)

## merge dataframes to have species-specific folder, reference folder and fastq ftps in same dataframe
species_and_refs = species_and_refs.merge(references, how = "left")

##########################################
#    	     --- Preparations ---
##########################################

for i in range(species_and_refs.shape[0]):
    # Initialising folders and variables for putting in the functions
    group      = species_and_refs.FOLDER[i]
    ref_folder = species_and_refs.REFERENCE_FOLDER[i]
    
    with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/parameters_{group}.pkl", "rb") as file:
        parameters = pickle.load(file)    
    with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/population_list.txt") as f:
        pops = [line.strip() for line in f]
    population_name = pops[0]
    df_regions = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/data/{ref_folder}/ref/regions_{ref_folder}_updated.txt", delimiter="\t")
    with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/{population_name}.txt") as f:
        samples = [line.strip() for line in f if line.strip()]

    df_stats = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/data/{group}/analysis_input/smcpp/masked_regions/mask_stats_{group}.txt")

    chromosomes = df_regions[
            (df_regions["female_ploidy"] == 2) &
            (df_regions["male_ploidy"] == 2) &
            (df_regions["end"] > 20000000)
        ]["region"].tolist()

    # Build a contig length dict from the regions df
    contig_lengths = df_regions.set_index("region")["end"].to_dict()

    df_stats = df_stats[["sample"] + chromosomes]
    # Divide each chromosome column by its contig lengths
    for chrom in df_stats.columns:
        if chrom in contig_lengths:
            df_stats[chrom] = df_stats[chrom] / contig_lengths[chrom]
    df_stats

    df_stats.to_csv(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/mask_stats_{group}.txt")

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pickle
import os

if os.getcwd().startswith("/home/lakrids"):
    path_prefix = "/home/lakrids/GenomeDK"
else:
    path_prefix = "/faststorage/project/"

genus_list = ["Loxodonta", "Elephas", "Boselaphus", "Panthera", "Rhinoceros", "Ceratotherium", "Diceros"]

data = pd.concat([pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/metadata/samples_{genus}.txt")
                  for genus in genus_list], ignore_index=True).reset_index(drop=True)
ref_folders = sorted(set(data.REFERENCE_FOLDER))
references = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/metadata/references.txt")
references = references.loc[references.REFERENCE_FOLDER.isin(ref_folders)].reset_index(drop=True)
species_and_refs = pd.DataFrame({
    "FOLDER": data.FOLDER,
    "REFERENCE_FOLDER": data.REFERENCE_FOLDER,
    "GVCF_FOLDER": [data.GENUS.iloc[j] + "_" + data.SPECIES.iloc[j] for j in range(data.shape[0])]
}).drop_duplicates().reset_index(drop=True)
species_and_refs = species_and_refs.merge(references, how="left")

for i in range(species_and_refs.shape[0]):
    group      = species_and_refs.FOLDER[i]
    ref_folder = species_and_refs.REFERENCE_FOLDER[i]

    with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/parameters_{group}.pkl", "rb") as file:
        parameters = pickle.load(file)
    with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/population_list.txt") as f:
        pops = [line.strip() for line in f]
    population_name = pops[0]

    df_regions = pd.read_table(f"{path_prefix}/megaFauna/sa_megafauna/data/{ref_folder}/ref/regions_{ref_folder}_updated.txt")
    with open(f"{path_prefix}/megaFauna/sa_megafauna/results/{group}/{population_name}.txt") as f:
        samples = [line.strip() for line in f if line.strip()]

    stats_path = f"{path_prefix}/megaFauna/sa_megafauna/data/{group}/analysis_input/smcpp/masked_regions/mask_stats_{group}.txt"
    if not os.path.exists(stats_path):
        print(f"Skipping {group} — mask stats not found")
        continue

    df_stats = pd.read_table(stats_path, index_col="sample")

    chromosomes = df_regions[
        (df_regions["female_ploidy"] == 2) &
        (df_regions["male_ploidy"] == 2) &
        (df_regions["end"] > 20_000_000)
    ]["region"].tolist()

    contig_lengths = df_regions.set_index("region")["end"].to_dict()

    # Keep only relevant chromosomes
    df_stats = df_stats[[c for c in chromosomes if c in df_stats.columns]]

    # Divide by contig length to get fractions
    for chrom in df_stats.columns:
        if chrom in contig_lengths:
            df_stats[chrom] = df_stats[chrom] / contig_lengths[chrom]

    # Row order: samples first, then mask type rows
    mask_rows  = [r for r in ["mappability", "coverage", "final_merged"] if r in df_stats.index]
    sample_rows = [r for r in samples if r in df_stats.index]
    df_plot = df_stats.loc[sample_rows + mask_rows]

    # ── Plot ──────────────────────────────────────────────────────
    n_rows = len(df_plot)
    n_cols = len(df_plot.columns)
    fig_w  = max(12, n_cols * 0.6)
    fig_h  = max(4,  n_rows * 0.5)

    # Fixed figure size regardless of number of rows/cols
    FIG_SIZE = 14  # fixed size in inches for all species

    fig, ax = plt.subplots(figsize=(FIG_SIZE, FIG_SIZE))
    #fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    # Separate colormaps for samples vs mask rows
    data_samples = df_plot.loc[sample_rows].values
    data_masks   = df_plot.loc[mask_rows].values
    combined     = df_plot.values

    im = ax.imshow(combined, aspect="auto", cmap="YlOrRd",
                   vmin=0, vmax=1, interpolation="none")

    # Draw a separator line between samples and mask rows
    sep = len(sample_rows) - 0.5
    ax.axhline(sep, color="white", linewidth=3)

    # Axis labels
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(df_plot.columns, rotation=90, fontsize=8)
    ax.set_yticks(range(n_rows))
    ax.set_yticklabels(df_plot.index, fontsize=9)

    # Scale font size based on number of rows
    cell_fontsize = max(4, min(8, int(80 / max(n_rows, n_cols))))

    # Annotate cells with values
    for row in range(n_rows):
        for col in range(n_cols):
            val = combined[row, col]
            color = "white" if val > 0.6 else "black"
            ax.text(col, row, f"{val:.2f}", ha="center", va="center",
                    fontsize=6, color=color)

    plt.colorbar(im, ax=ax, label="Fraction masked", shrink=0.6)
    ax.set_title(f"{group} — masking fractions per chromosome", fontsize=12, pad=12)
    plt.tight_layout()

    out_path = f"{path_prefix}/megaFauna/sa_megafauna/results/shared/mask_stats/mask_stats_{group}_{population_name}_heatmap.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {out_path}")

Saved: /home/lakrids/GenomeDK/megaFauna/sa_megafauna/results/shared/mask_stats/mask_stats_Loxodonta_africana_pop1_heatmap.png
Saved: /home/lakrids/GenomeDK/megaFauna/sa_megafauna/results/shared/mask_stats/mask_stats_Loxodonta_cyclotis_pop0_heatmap.png
Saved: /home/lakrids/GenomeDK/megaFauna/sa_megafauna/results/shared/mask_stats/mask_stats_Elephas_maximus_pop0_heatmap.png
Saved: /home/lakrids/GenomeDK/megaFauna/sa_megafauna/results/shared/mask_stats/mask_stats_Boselaphus_tragocamelus_pop0_heatmap.png
Saved: /home/lakrids/GenomeDK/megaFauna/sa_megafauna/results/shared/mask_stats/mask_stats_Panthera_leo_pop0_heatmap.png
Saved: /home/lakrids/GenomeDK/megaFauna/sa_megafauna/results/shared/mask_stats/mask_stats_Panthera_pardus_pop1_heatmap.png
Saved: /home/lakrids/GenomeDK/megaFauna/sa_megafauna/results/shared/mask_stats/mask_stats_Panthera_tigris_pop0_heatmap.png
Saved: /home/lakrids/GenomeDK/megaFauna/sa_megafauna/results/shared/mask_stats/mask_stats_Panthera_uncia_pop0_heatmap.png
Saved: